<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 6 — Your 50 Stocks for 2026, and How Often to Rebalance

In Studio 5 you built a fund with Ms. Womack's model and your own business rules, and it is on the class
leaderboard. Tonight you build version 2:

* **Warm-up · Seven ways to pick 50 stocks** on Dec 31, 2025, with no peeking at 2026. Which way would you bet on?
* **Part 1 · Your stocks for 2026**, with **daily** prices this time (Studio 5 used monthly).
* **Part 2 · Ms. Womack with a memory.** The model only looks back a set number of trading days.
* **Part 3 · How often to rebalance.** Weekly or monthly? Re-fit the model, or just trade back to your starting
  mix? With and without your rules, and with and without a small trading cost.
* **Part 4 · Put your best strategy on the leaderboard**, with its 2026 backtest.

All of the code is written. You change the dictionaries. When you see **Predict first**, write down your guess
before you run the next cell.

## Import modules (download Pyomo, ipopt, Bonmin and yfinance)

Same setup as Studio 5. Run this first, every time. About a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt and Bonmin)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
import time
from pyomo.environ import *
from IPython.display import display

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solvers
ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

# Warm-up · Seven ways to pick 50 stocks on Dec 31, 2025

**The rule of the game: no peeking.** Pretend it is the evening of Dec 31, 2025. You can know:

* who was in the S&P 500 that day (we use the list as Wikipedia showed it in late December 2025, not today's list),
* how big each company was that day (its market value: shares times the Dec 31 price),
* what each stock did during 2025.

You cannot know anything about 2026. Picking stocks with today's list, or today's biggest companies, is peeking.

**One honest wrinkle.** A few of those companies were bought or renamed during 2026, and Yahoo Finance no longer has
their prices. They are missing from our table. Data that quietly drops the companies that disappeared is called
**survivorship bias**: it makes the past look a little better than it was.

In [ ]:
members = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/sp500_2025-12-31.csv')
print(len(members), "companies in the S&P 500 on Dec 31, 2025 (with a full year of 2025 prices)")
print()
print(members['sector'].value_counts().to_string())
members.sort_values('market_value_billions', ascending=False).head(10)

### The seven ways (plus two more)

| `way` | The 50 stocks are... | The idea |
|---|---|---|
| `'biggest'` | the 50 largest companies by market value | big companies are steady |
| `'sector_balanced'` | the 5 largest in each sector, then the 50 largest of those | spread across the economy |
| `'most_traded'` | the 50 with the most dollars traded per day in 2025 | where the action is |
| `'winners_2025'` | 2025's 50 best stocks | winners keep winning (momentum) |
| `'losers_2025'` | 2025's 50 worst stocks | buy the dip |
| `'calmest'` | the 50 with the smallest daily ups and downs in 2025 | sleep at night |
| `'return_per_risk'` | the 50 with the best average day per unit of ups and downs | what Ms. Womack likes |
| `'random'` | 50 at random | the baseline everything should beat |
| `'mine'` | your own list | your interests |

In [ ]:
def pick_stocks(way, how_many=50, my_picks=None, seed=5641):
    table = members.copy()
    if way == 'biggest':
        picked = table.sort_values('market_value_billions', ascending=False).head(how_many)
    elif way == 'sector_balanced':
        picked = pd.DataFrame()
        for sector in sorted(table['sector'].unique()):
            in_sector = table[table['sector'] == sector].sort_values('market_value_billions', ascending=False)
            picked = pd.concat([picked, in_sector.head(5)])
        picked = picked.sort_values('market_value_billions', ascending=False).head(how_many)
    elif way == 'most_traded':
        picked = table.sort_values('dollars_traded_per_day_2025_millions', ascending=False).head(how_many)
    elif way == 'winners_2025':
        picked = table.sort_values('return_2025_pct', ascending=False).head(how_many)
    elif way == 'losers_2025':
        picked = table.sort_values('return_2025_pct').head(how_many)
    elif way == 'calmest':
        picked = table.sort_values('daily_ups_and_downs_2025_pct').head(how_many)
    elif way == 'return_per_risk':
        picked = table.sort_values('return_per_risk_2025', ascending=False).head(how_many)
    elif way == 'random':
        picked = table.sample(how_many, random_state=seed)
    elif way == 'mine':
        picked = table[table['ticker'].isin(my_picks)]
    else:
        print("Unknown way:", way)
        picked = pd.DataFrame()
    return picked


example = pick_stocks('winners_2025')
print("2025's 50 biggest winners, by sector:")
print(example['sector'].value_counts().to_string())
example[['ticker', 'company', 'sector', 'return_2025_pct']].head(10)

**Predict first.** Put \$10,000 into each way on Dec 31, 2025, split evenly across its 50 stocks, and hold it. Which way
ends 2026 (so far) with the most money? Which ends with the least? Does any of them beat the S&P 500?

The next cell uses the class copy of real daily prices (Oct 2024 to early November 2026).

In [ ]:
saved_prices = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/daily_prices.csv', index_col=0, parse_dates=True)
since = saved_prices.loc['2025-12-31':]

ways = ['biggest', 'sector_balanced', 'most_traded', 'winners_2025', 'losers_2025', 'calmest', 'return_per_risk']
warm_up = {}
for way in ways:
    tickers = []
    for t in pick_stocks(way)['ticker']:
        if t in since.columns:
            tickers.append(t)
    growth = since[tickers].ffill() / since[tickers].iloc[0]
    warm_up[way] = 10000 * growth.mean(axis=1).iloc[-1]

# 200 random sets of 50, as a yardstick
random_values = []
for seed in range(200):
    tickers = []
    for t in pick_stocks('random', seed=seed)['ticker']:
        if t in since.columns:
            tickers.append(t)
    growth = since[tickers].ffill() / since[tickers].iloc[0]
    random_values.append(10000 * growth.mean(axis=1).iloc[-1])
spy_value = 10000 * since['SPY'].iloc[-1] / since['SPY'].iloc[0]

warm_table = pd.Series(warm_up).sort_values()
figure(figsize=(10, 4.5))
barh(warm_table.index, warm_table.values, color='tab:blue', label='equal split of its 50 stocks')
axvspan(np.percentile(random_values, 5), np.percentile(random_values, 95), color='lightgray', alpha=0.6, zorder=0,
        label='90% of random 50s')
axvline(spy_value, color='black', linewidth=2, label='S&P 500 (SPY)')
xlim(min(9000, warm_table.min() - 500), max(warm_table.max(), spy_value) + 500)
xlabel('Value of $10,000 invested on Dec 31, 2025')
title('Seven ways to pick 50 stocks, held from Dec 31, 2025 to ' + str(since.index[-1].date()))
legend(loc='lower right')
show()
print("S&P 500: $%.0f   random 50s: middle 90%% from $%.0f to $%.0f" % (
    spy_value, np.percentile(random_values, 5), np.percentile(random_values, 95)))
print(warm_table.round(0).to_string())

**What to notice.** How many of the seven ways land inside the gray band, where 50 random stocks usually land?
How many energy stocks does each way hold? 2026 was a big year for oil companies (Exxon and Chevron were up about
50% by October). Did any rule you could have used on Dec 31 see that coming?

# Part 1 · Your stocks for 2026

### Step 1 · Choose your way, and how many stocks

Pick one of the ways above, or `'mine'` with your own list. Between 20 and 50 stocks works well tonight (more stocks
means more yes/no variables for Bonmin, so the rules take longer). For `'mine'`, use tickers from the table above;
the Studio 5 starter list is filled in as an example.

In [ ]:
my_way = 'biggest'      # 'biggest', 'sector_balanced', 'most_traded', 'winners_2025', 'losers_2025',
                        # 'calmest', 'return_per_risk', 'random' or 'mine'
how_many = 50           # 20 to 50
my_picks = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'JPM', 'GS', 'V', 'JNJ', 'LLY', 'UNH', 'XOM', 'CVX',
            'KO', 'WMT', 'COST', 'AMZN', 'HD', 'CAT', 'LMT']

chosen = pick_stocks(my_way, how_many, my_picks)
sector_of = {}
for i in range(len(chosen)):
    sector_of[chosen['ticker'].iloc[i]] = chosen['sector'].iloc[i]

print(len(sector_of), "stocks, picked by:", my_way)
print(chosen['sector'].value_counts().to_string())

### Step 2 · Download daily prices

Daily, adjusted closing prices from October 2024 (so the model has a year of memory before Jan 2, 2026) up to the
latest close. Same safety net as Studio 5: if Yahoo blocks Colab, the cell uses the class copy.

In [ ]:
import yfinance as yf

start = '2024-10-01'
prices = pd.DataFrame()
failed = []
for ticker in list(sector_of) + ['SPY']:
    try:
        data = yf.download(ticker, start=start, interval='1d', auto_adjust=True, progress=False)
        if len(data) > 250:
            prices[ticker] = data['Close'].squeeze()
        else:
            failed.append(ticker)
    except Exception as problem:
        failed.append(ticker)

if len(failed) > 0:
    print("These did not download:", failed)

# the fallback: the class copy of real daily prices
if 'SPY' not in prices.columns or len(prices.columns) < 0.8 * (len(sector_of) + 1):
    print("Yahoo did not give us enough stocks, so we are using the class copy instead.")
    prices = pd.DataFrame()
    for ticker in list(sector_of) + ['SPY']:
        if ticker in saved_prices.columns:
            prices[ticker] = saved_prices[ticker]

# today's row is not a close until the market has closed, so we stop at yesterday
today = pd.Timestamp.today().normalize()
prices = prices[prices.index < today]
print(prices.shape[0], "trading days for", prices.shape[1] - 1, "stocks + SPY, through", prices.index[-1].date())

### Step 3 · Prices into daily returns, then check them

$$r_t = \frac{P_t}{P_{t-1}} - 1 \qquad \text{(now } t \text{ is a trading day, not a month)}$$

In [ ]:
returns = prices.pct_change().iloc[1:]

# drop any stock with missing days (a ticker change, a late listing)
stocks = []
for ticker in sector_of:
    if ticker in returns.columns and returns[ticker].isna().sum() == 0:
        stocks.append(ticker)
    else:
        print("dropping", ticker)
spy_returns = returns['SPY']
returns = returns[stocks]

print("trading days:", len(returns))
print("stocks:", len(stocks))
moves = pd.DataFrame({'worst day': returns.min(), 'best day': returns.max(), 'average day': returns.mean()})
moves.sort_values('worst day').round(3).head(8)

**Look before you model.** A daily move worse than -40% is usually a data problem, not a crash. Compare the
average day here with the average month in Studio 5: a month is about 21 trading days.

# Part 2 · Ms. Womack with a memory

Same model as Studio 5, with two changes:

1. **Daily returns.** The average return and the covariance come from daily data.
2. **A memory.** Each time we fit the model, it only sees the last `memory_days` trading days before that day.
   About 63 trading days is 3 months; 252 is a year.

**The risk limit, every time we fit:** the risk of an even split of your stocks over the same days. In words: *be as
risky as simply spreading the money evenly, but earn the most you can for that risk.* (It is Studio 5's Step 6
yardstick. We use it here because finding the top-left corner at every refit would mean solving a whole frontier
each week.)

### Step 4 · Your rules

Same rules as Studio 5. The sector names now follow the official S&P list (so technology is
`'Information Technology'`). If your stocks have no stock in a must-have sector, that rule is skipped with a note.

In [ ]:
rules = {
    'min_share': 0.02,                                                        # if you invest at all, at least 2%
    'max_share': 0.10,                                                        # never more than 10% in one stock
    'must_have_sectors': ['Information Technology', 'Health Care', 'Energy'],  # at least one stock from each
    'max_stocks': None,                                                       # e.g. 25, or None for no limit
}

for sector in rules['must_have_sectors']:
    count = 0
    for s in stocks:
        if sector_of[s] == sector:
            count += 1
    if count == 0:
        print("Note: none of your stocks is in", sector, "- that rule will be skipped")
if len(stocks) * rules['max_share'] < 1:
    print("Not enough stocks for the cap: add stocks or raise max_share.")
else:
    print("OK:", len(stocks), "stocks")

### Step 5 · The model, with a switch for the rules

`solve_portfolio(window, use_rules)` is Studio 5's `solve_portfolio` with one change: it gets a **window** of daily
returns (the model's memory) instead of the whole history.

In [ ]:
def solve_portfolio(window, use_rules):
    df_return = window.mean()
    df_cov = window.cov()

    # the risk limit: the risk of an even split over the same days
    risk_limit = 0
    for i in stocks:
        for j in stocks:
            risk_limit += df_cov.loc[i, j] / len(stocks) ** 2

    model = ConcreteModel()
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    if use_rules:
        model.y = Var(stocks, domain=Binary)
        for s in stocks:
            model.y[s] = 1
        for s in stocks:
            model.constraints.add(model.x[s] <= rules['max_share'] * model.y[s])
            if rules['min_share'] is not None:
                model.constraints.add(model.x[s] >= rules['min_share'] * model.y[s])
        for sector in rules['must_have_sectors']:
            picks = 0
            count = 0
            for s in stocks:
                if sector_of[s] == sector:
                    picks += model.y[s]
                    count += 1
            if count > 0:
                model.constraints.add(picks >= 1)
        if rules['max_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held <= rules['max_stocks'])
        solver = SolverFactory('bonmin', executable=bonmin_path)
        solver.options['bonmin.time_limit'] = 60
        results = solver.solve(model)
    else:
        results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    if results.solver.termination_condition not in [TerminationCondition.optimal, TerminationCondition.maxTimeLimit]:
        return None
    weights = {}
    for s in stocks:
        share = model.x[s]()
        if share is None or share < 0.0001:
            share = 0
        weights[s] = share
    total = 0
    for s in stocks:
        total += weights[s]
    for s in stocks:
        weights[s] = weights[s] / total
    return weights


def window_before(day, memory_days):
    # the last memory_days trading days BEFORE day (the model never sees day itself)
    return returns[returns.index < day].tail(memory_days)


print("solve_portfolio is ready")

### Step 6 · The Jan 2, 2026 portfolio, with a 1-year memory

**Predict first.** With a year of memory, how much of the money goes into Energy with no rules? With your rules?

In [ ]:
first_day = returns.loc['2026-01-01':].index[0]
start_window = window_before(first_day, 252)

t0 = time.time()
jan2_free = solve_portfolio(start_window, use_rules=False)
jan2_ruled = solve_portfolio(start_window, use_rules=True)
print("solved in %.1f seconds" % (time.time() - t0))

jan2 = pd.DataFrame({'no rules (%)': pd.Series(jan2_free) * 100, 'with your rules (%)': pd.Series(jan2_ruled) * 100})
jan2 = jan2[(jan2['no rules (%)'] > 0.05) | (jan2['with your rules (%)'] > 0.05)]
jan2['sector'] = pd.Series(sector_of)
print("stocks held: no rules", int((jan2['no rules (%)'] > 0.05).sum()), "| with your rules", int((jan2['with your rules (%)'] > 0.05).sum()))
print("in Energy:   no rules %.1f%% | with your rules %.1f%%" % (
    jan2[jan2['sector'] == 'Energy']['no rules (%)'].sum(), jan2[jan2['sector'] == 'Energy']['with your rules (%)'].sum()))
jan2.sort_values('with your rules (%)', ascending=False).round(1)

# Part 3 · How often to rebalance

Every portfolio starts on Jan 2, 2026 with \$10,000. Between trades it is **bought and held**: each stock's share
drifts as its price moves. A **rebalance** is a day when we trade. There are two kinds:

| Method | On a rebalance day... | Uses the model again? |
|---|---|---|
| `'refit'` | re-fit Ms. Womack on the latest `memory_days` and trade to the new mix | yes |
| `'back to start'` | trade back to the Jan 2 mix (sell what grew, buy what shrank) | no |
| `'buy and hold'` | never trade after Jan 2 | no |

**How often:** weekly (every 5 trading days) or monthly (every 20).

**Trading costs:** every dollar we buy or sell costs 0.1% (a stand-in for fees and the bid-ask spread). Trade
\$4,000 worth and you pay \$4. We run everything with and without the cost.

### Step 7 · The experiment

In [ ]:
experiment = {
    'methods': ['refit', 'back to start', 'buy and hold'],
    'memory_days': [63, 252],                    # about 3 months and about 1 year (refit only)
    'every_days': {'weekly': 5, 'monthly': 20},  # how often to rebalance
    'trading_cost': 0.001,                       # 0.1% of every dollar traded
}

### Step 8 · The engine

`run_strategy` walks through 2026 one trading day at a time. On each rebalance day it trades to the target mix and
pays the trading cost; on the other days the money just rides the prices. `targets_for` does the solving, and it
remembers every answer, so the monthly runs reuse the weekly solves (20 is a multiple of 5).

In [ ]:
days = returns.loc['2026-01-01':].index
solved = {}


def targets_for(day, memory_days, use_rules):
    # the model's mix for one rebalance day (solved once, then remembered)
    key = (day, memory_days, use_rules)
    if key not in solved:
        weights = solve_portfolio(window_before(day, memory_days), use_rules)
        if weights is None:
            weights = solve_portfolio(window_before(day, memory_days), False)
        solved[key] = weights
    return solved[key]


def run_strategy(method, memory_days, every, use_rules, cost):
    start_mix = targets_for(days[0], memory_days, use_rules)
    value = 10000.0
    holdings = {}
    traded = 0.0
    for s in stocks:
        holdings[s] = value * start_mix[s]
        traded += holdings[s]
    value = value - cost * traded
    for s in stocks:
        holdings[s] = value * start_mix[s]

    path = {days[0]: value}
    shares_over_time = {days[0]: start_mix}
    for k in range(1, len(days)):
        day = days[k]
        value = 0
        for s in stocks:
            holdings[s] = holdings[s] * (1 + returns.loc[day, s])
            value += holdings[s]
        rebalance_today = method != 'buy and hold' and k % every == 0
        if rebalance_today:
            if method == 'refit':
                target = targets_for(day, memory_days, use_rules)
            else:
                target = start_mix
            trade = 0
            for s in stocks:
                trade += abs(value * target[s] - holdings[s])
            traded += trade
            value = value - cost * trade
            for s in stocks:
                holdings[s] = value * target[s]
            shares_over_time[day] = target
        path[day] = value
    return {'path': pd.Series(path), 'traded': traded, 'shares': pd.DataFrame(shares_over_time).T}


print("2026 so far:", len(days), "trading days, from", days[0].date(), "to", days[-1].date())

### Step 9 · Run it

**Predict first.** Write down your guesses:

1. Weekly or monthly: which does better?
2. Refit or back to start?
3. A 3-month memory or a 1-year memory?
4. Does the trading cost change the winner?

With 50 stocks the rules take Bonmin a second or two per solve, so this cell takes a few minutes. It prints as it goes.

In [ ]:
t0 = time.time()
rows = []
runs = {}
for use_rules in [False, True]:
    for method in experiment['methods']:
        if method == 'refit':
            memories = experiment['memory_days']
        else:
            memories = [252]
        if method == 'buy and hold':
            schedules = {'never': 1}
        else:
            schedules = experiment['every_days']
        for memory_days in memories:
            for schedule in schedules:
                for cost in [0.0, experiment['trading_cost']]:
                    result = run_strategy(method, memory_days, schedules[schedule], use_rules, cost)
                    path = result['path']
                    if method == 'refit':
                        name = 'refit %s, %d-day memory' % (schedule, memory_days)
                    elif method == 'back to start':
                        name = 'back to start %s' % schedule
                    else:
                        name = 'buy and hold'
                    runs[(name, use_rules, cost)] = result
                    rows.append({
                        'strategy': name,
                        'rules': 'with your rules' if use_rules else 'no rules',
                        'trading cost': 'with cost' if cost > 0 else 'no cost',
                        'ending value ($)': round(path.iloc[-1], 0),
                        'dollars traded ($)': round(result['traded'], 0),
                        'worst drop (%)': round(100 * (1 - path / path.cummax()).max(), 1),
                    })
        print("done:", 'with your rules' if use_rules else 'no rules', "/", method, "(%.0f seconds so far)" % (time.time() - t0))

results = pd.DataFrame(rows)
spy_path = 10000 * (1 + spy_returns.loc[days]).cumprod() / (1 + spy_returns.loc[days[0]])
print()
print("S&P 500 (SPY) over the same days: $%.0f" % spy_path.iloc[-1])
table = results.pivot_table(index='strategy', columns=['rules', 'trading cost'], values='ending value ($)')
table.sort_values(('with your rules', 'with cost'), ascending=False)

**Read the table.** Each row is a strategy; each column is rules (or not) and cost (or not). Find your four
predictions in it. Which was right? Then look at how many dollars each strategy traded: that is what the cost is
charged on.

In [ ]:
traded = results[results['trading cost'] == 'with cost'].pivot_table(index='strategy', columns='rules', values='dollars traded ($)')
traded.sort_values('with your rules', ascending=False)

### The race, with your rules and the trading cost

In [ ]:
line_styles = {
    'refit weekly, 63-day memory': ('tab:red', '-'),
    'refit monthly, 63-day memory': ('tab:red', '--'),
    'refit weekly, 252-day memory': ('tab:green', '-'),
    'refit monthly, 252-day memory': ('tab:green', '--'),
    'back to start weekly': ('tab:blue', '-'),
    'back to start monthly': ('tab:blue', '--'),
    'buy and hold': ('gray', '-'),
}
figure(figsize=(12, 5.5))
for name in line_styles:
    color, style = line_styles[name]
    path = runs[(name, True, experiment['trading_cost'])]['path']
    plot(path.index, path.values, color=color, linestyle=style, label=name)
plot(spy_path.index, spy_path.values, color='black', linewidth=2.5, label='S&P 500 (SPY)')
axhline(10000, color='lightgray', linestyle=':')
ylabel('Value of $10,000')
title('2026 so far, with your rules, after a 0.1% trading cost')
legend(fontsize=8, loc='upper left')
show()

### Did oil sneak in?

The share of the money in Energy at every rebalance, for the two refit memories (weekly, with your rules). The gray
line is how much Exxon and Chevron themselves were up in 2026 (right axis), if they are among your stocks.

In [ ]:
energy = []
for s in stocks:
    if sector_of[s] == 'Energy':
        energy.append(s)

if len(energy) == 0:
    print("None of your stocks is in Energy, so oil could not sneak in. Try my_way = 'sector_balanced'.")
else:
    fig, axis = subplots(figsize=(12, 4.5))
    for memory_days, color in [(63, 'tab:red'), (252, 'tab:green')]:
        shares = runs[('refit weekly, %d-day memory' % memory_days, True, experiment['trading_cost'])]['shares']
        in_energy = shares[energy].sum(axis=1) * 100
        axis.step(in_energy.index, in_energy.values, where='post', color=color, label='%d-day memory' % memory_days)
    axis.set_ylabel('% of the money in Energy')
    axis.legend(loc='upper left')
    oil = []
    for t in ['XOM', 'CVX']:
        if t in prices.columns:
            oil.append(t)
    if len(oil) > 0:
        twin = axis.twinx()
        both = prices.loc[days, oil].mean(axis=1)
        twin.plot(both.index, 100 * (both / both.iloc[0] - 1), color='gray', alpha=0.5)
        twin.set_ylabel(' + '.join(oil) + ' since Jan 2 (%)')
    title('Share in Energy, refit weekly, with your rules (' + str(len(energy)) + ' energy stocks to choose from)')
    show()

**What to notice.** The short memory reacts fast: it piles into whatever just went up, and it is often late. The
long memory moves slowly. When oil jumped in March 2026, which one bought near the top?

### Who did we hold, and when?

Each row is a stock, each column a rebalance day; darker means a bigger share. Change `show_strategy` to look at
another one.

In [ ]:
show_strategy = 'refit weekly, 252-day memory'

shares = runs[(show_strategy, True, experiment['trading_cost'])]['shares'] * 100
biggest = shares.max().sort_values(ascending=False).head(20).index
figure(figsize=(12, 6))
imshow(shares[biggest].T.values, aspect='auto', cmap='Oranges', vmin=0, vmax=shares.values.max())
yticks(range(len(biggest)), biggest)
step = max(1, len(shares) // 10)
xticks(range(0, len(shares), step), shares.index[::step].strftime('%b %d'))
colorbar(label='% of the money')
title(show_strategy + ', with your rules: the 20 stocks it used most')
show()

# Part 4 · Put your best strategy on the leaderboard

### Step 10 · Choose your strategy

Pick the strategy you believe in for the **rest of** 2026, not just the one that won the past ten months. Your fund
is the version **with your rules**; the leaderboard tracks its **no-rules twin** right next to it.

In [ ]:
my_strategy = {
    'method': 'refit',          # 'refit', 'back to start' or 'buy and hold'
    'schedule': 'monthly',      # 'weekly' or 'monthly' (ignored for 'buy and hold')
    'memory_days': 252,         # 63 or 252 (used by 'refit'; the starting mix always uses 252)
}

if my_strategy['method'] == 'refit':
    strategy_name = 'refit %s, %d-day memory' % (my_strategy['schedule'], my_strategy['memory_days'])
elif my_strategy['method'] == 'back to start':
    strategy_name = 'back to start %s' % my_strategy['schedule']
else:
    strategy_name = 'buy and hold'

ruled_run = runs[(strategy_name, True, experiment['trading_cost'])]
free_run = runs[(strategy_name, False, experiment['trading_cost'])]
print("your strategy:", strategy_name)
print("2026 so far, after costs: with your rules $%.0f, no rules $%.0f, S&P 500 $%.0f" % (
    ruled_run['path'].iloc[-1], free_run['path'].iloc[-1], spy_path.iloc[-1]))

### Step 11 · What your fund holds from now on

The leaderboard holds whatever you send until you send again. So we send what your strategy holds **today**:

* `'refit'`: re-fit the model on the latest `memory_days` (through the last close),
* `'back to start'`: the Jan 2 mix (that is what it trades back to),
* `'buy and hold'`: the Jan 2 mix after ten months of drifting.

In [ ]:
def holdings_today(use_rules):
    if my_strategy['method'] == 'refit':
        return solve_portfolio(returns.tail(my_strategy['memory_days']), use_rules)
    start_mix = targets_for(days[0], 252, use_rules)
    if my_strategy['method'] == 'back to start':
        return start_mix
    grown = {}
    total = 0
    for s in stocks:
        grown[s] = start_mix[s] * prices[s].iloc[-1] / prices.loc[days[0], s]
        total += grown[s]
    for s in stocks:
        grown[s] = grown[s] / total
    return grown


now_ruled = holdings_today(True)
now_free = holdings_today(False)
now = pd.DataFrame({'no rules (%)': pd.Series(now_free) * 100, 'with your rules (%)': pd.Series(now_ruled) * 100})
now = now[(now['no rules (%)'] > 0.05) | (now['with your rules (%)'] > 0.05)]
now['sector'] = pd.Series(sector_of)
now.sort_values('with your rules (%)', ascending=False).round(1)

### Step 12 · Send it

Same as Studio 5: fill in the lines, run the next cell, sign in when Colab asks. Your fund page on the leaderboard
will show this strategy's 2026 backtest (with your rules, its no-rules twin, an even split and the S&P 500).

This counts as a **change** to your Studio 5 fund: it starts from what your fund is worth at the next close, and the
reason is saved in your history. Use the same NetID and nickname as in Studio 5.

In [ ]:
netid = ""            # your UConn NetID, for example abc12345
nickname = ""         # the SAME nickname as in Studio 5
theme = ""            # one line: what your fund is about now
class_code = ""       # Dr. Dave writes it on the board

reason = "Studio 6: %s, %d stocks picked by %s" % (strategy_name, len(stocks), my_way)
print("reason:", reason)

In [ ]:
import json
import requests

SUBMIT_URL = 'https://us-central1-opim5641-portfolio-f26.cloudfunctions.net/submit-portfolio'
BOARD_URL = 'https://storage.googleapis.com/opim5641-leaderboard-f26/index.html'


def shares_to_send(weights):
    kept = {}
    for s in stocks:
        if weights[s] > 0.001:
            kept[s] = round(float(weights[s]), 6)
    return kept


# the 2026 backtest, one point a week (small enough to send)
even_path = 10000 * (prices.loc[days, stocks] / prices.loc[days[0], stocks]).mean(axis=1)
lines = {'with your rules': ruled_run['path'], 'no rules': free_run['path'],
         'equal split of your stocks': even_path, 'S&P 500 (SPY)': spy_path}
backtest_paths = {}
backtest_table = {}
for name in lines:
    path = lines[name]
    one_line = {}
    for k in range(0, len(path), 5):
        one_line[str(path.index[k].date())] = round(float(path.iloc[k]), 2)
    one_line[str(path.index[-1].date())] = round(float(path.iloc[-1]), 2)
    backtest_paths[name] = one_line
    backtest_table[name] = {'ending value ($)': round(float(path.iloc[-1]), 0),
                            'total return (%)': round(100 * float(path.iloc[-1] / 10000 - 1), 1),
                            'worst drop (%)': round(100 * float((1 - path / path.cummax()).max()), 1)}

entry = {
    'netid': netid.strip().lower(),
    'nickname': nickname.strip(),
    'theme': theme.strip(),
    'reason': reason.strip(),
    'class_code': class_code.strip(),
    'portfolios': {'no_rules': shares_to_send(now_free), 'with_rules': shares_to_send(now_ruled)},
    'rules': rules,
    'strategy': {'name': strategy_name, 'way': my_way, 'stocks': len(stocks), 'trading_cost': experiment['trading_cost']},
    'sectors': sector_of,
    'data_window': [str(returns.index[0].date()), str(returns.index[-1].date())],
    'backtest': backtest_table,
    'backtest_paths': backtest_paths,
    'backtest_title': "2026 so far: %s, $10,000 from Jan 2, after a 0.1%% trading cost" % strategy_name,
    'backtest_note': "From the Studio 6 notebook: the strategy %s, run on every trading day of 2026 so far, "
                     "starting with $10,000 on Jan 2 and paying a 0.1%% trading cost. The model only ever used data "
                     "from before each rebalance day." % strategy_name,
    'notebook': 'studio6',
}

print("What you are sending (share of the money in each stock):")
print(now.sort_values('with your rules (%)', ascending=False).round(1).drop(columns='sector').to_string())
print()

missing = []
for field in ['netid', 'nickname', 'theme', 'class_code']:
    if entry[field] == '':
        missing.append(field)

if len(missing) > 0:
    print("Not sent yet. Fill in:", ", ".join(missing), "(in the cell above), then run this cell again.")
elif 'google.colab' not in sys.modules:
    print("Not sent: run this notebook in Google Colab to sign in and submit.")
else:
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    answer = requests.post(SUBMIT_URL, data=json.dumps(entry, default=str), timeout=60,
                           headers={'Authorization': 'Bearer ' + credentials.token, 'Content-Type': 'application/json'})
    receipt = answer.json()
    if receipt.get('ok'):
        received = pd.Timestamp(receipt['received_utc']).tz_convert('America/New_York')
        print("Your fund is in! This is what the leaderboard SAVED. Check that it is the model you meant to send.")
        print("  nickname:     ", receipt['nickname'])
        print("  NetID:        ", receipt['netid'], " (signed in as", receipt['google_email'] + ")")
        print("  theme:        ", receipt.get('theme', ''))
        print("  reason:       ", receipt.get('reason', ''))
        print("  received:     ", received.strftime('%a %b %d, %I:%M %p'), "Eastern")
        print("  stocks held:  ", receipt['holdings']['with_rules'], "with your rules,", receipt['holdings']['no_rules'], "with no rules")
        print("  it counts from the next market close (4 PM Eastern on a weekday)")
        if receipt.get('fund_page'):
            print("  your fund's page:", receipt['fund_page'], "(ready within 5 minutes)")
        print("  the leaderboard: ", BOARD_URL, "(updates every 5 minutes)")
    else:
        print("Not accepted:", receipt.get('problem'))

### Keep it running

Your strategy has a schedule. To follow it, open this notebook on that schedule (every week, or every month), choose
the same way, rules and strategy, **Run all**, and send again. Each send is saved with its time and reason, so your
fund page shows exactly when you rebalanced. That is the whole job of a fund manager who follows a rule: do the same
thing, on time, and write down why.

A fund can be updated on more than one clock. The **memory** (how far back the model looks) and the **schedule**
(how often you trade) are two different clocks, and you can update a model daily, weekly, monthly or quarterly.
Many real funds re-fit their models monthly or quarterly and only trade back to their targets in between.

### Your turn

Answer in a text cell:

1. Which of the seven ways did best in the warm-up? Would you have bet on it on Dec 31, 2025? Why or why not?
2. Which of your four predictions in Step 9 were right?
3. How many dollars did your weekly refit trade compared with buy and hold? What did that cost?
4. Did oil sneak into your portfolio? With which memory, and when?
5. Which strategy did you send, and why do you believe in it for the rest of 2026?

### Where this is going: the final project

Your 2-page report compares **Ms. Womack with no rules** against **your rules**, and **how often you rebalanced**
(and how you managed it). Everything you need is on your fund page and in this notebook:

* your fund and its no-rules twin, every day since Studio 5, against the S&P 500,
* your history of changes, with the reasons you wrote,
* your Studio 6 backtest: rules vs no rules, rebalancing schedule, with and without costs.

Keep your fund running until the report is due. A strategy you stuck with tells a better story than one you
abandoned.